# IndoML T2 — FREE CHECK of the 'reference keeps native noise' hypothesis (Kaggle T4, ~40 min)
Build natural clip + library events (target = natural clip) and score D. If D's SI-SDR here ≈ LB (13 dB) while validation synthetic gives ~18 dB, the hypothesis holds and E13 is worth the money.
Inputs: `indoml-validation`, `indoml-t2-ckpt`, `musan-noise`, `mpsenet-e10r`. GPU T4, Internet ON, HF_TOKEN ON.

In [ ]:
!rm -rf /kaggle/working/ner && git clone -q --depth 1 https://github.com/nagshubhadip/noise-event-detection-and-removal /kaggle/working/ner
!pip -q install jiwer sentencepiece webrtcvad-wheels librosa 2>&1 | tail -1
!wget -q https://github.com/karolpiczak/ESC-50/archive/refs/heads/master.zip -O /kaggle/working/esc50.zip && unzip -q -o /kaggle/working/esc50.zip -d /kaggle/working/ && ls /kaggle/working/ESC-50-master | head
# ---- MP-SENet (MIT) ----
!rm -rf /kaggle/working/mpsenet && git clone -q --depth 1 https://github.com/yxlu-0102/MP-SENet /kaggle/working/mpsenet
!pip -q install einops 2>&1 | tail -1

In [ ]:
!pip -q install datasets 2>&1 | tail -1

In [ ]:
import sys, os, glob, json, time, random, numpy as np, torch, soundfile as sf, librosa
from pathlib import Path
VAL = Path('/kaggle/input/datasets/avin55/indoml-validation/validation')
M = Path('/kaggle/working/ner/track2_removal/versions/v-1_indoml2026_track2/modular')
d = (M/'data.py').read_text()
d = d.replace('Path("/kaggle/input/indoml-validation"),', 'Path("/kaggle/input/datasets/avin55/indoml-validation/validation"), Path("/kaggle/input/indoml-validation"),')
d = d.replace('cid = (r.get("clip_id") or r.get("id")', 'cid = (r.get("segmentFileName") or r.get("clip_id") or r.get("id")')
(M/'data.py').write_text(d); sys.path.insert(0, str(M))
from kaggle_secrets import UserSecretsClient
HF = UserSecretsClient().get_secret("HF_TOKEN"); os.environ["HF_TOKEN"] = HF
import config, data, evaluate
import model as model_mod
from train import train_on_sim
from inference import enhance, Transcriber
from jiwer import wer as jwer
config.ensure_dirs()
val_dir, items = data.resolve_validation()
meta = {m['segmentFileName'][:-4]: m for m in json.load(open(VAL/'validationMetadata.json', encoding='utf-8'))}
print('validation clips', len(items))
import torch.nn.functional as F

In [ ]:
import types, sys
sys.modules.setdefault('pesq', types.SimpleNamespace(pesq=lambda *a, **k: 0.0))   # PESQ not needed (no metric discriminator)
sys.path.insert(0, '/kaggle/working/mpsenet')
from env import AttrDict
from dataset import mag_pha_stft, mag_pha_istft
from models.model import MPNet, phase_losses
h = AttrDict(json.load(open('/kaggle/working/mpsenet/best_ckpt/config.json')))
dev = torch.device('cuda')
def load_mp(path='/kaggle/working/mpsenet/best_ckpt/g_best_dns'):
    m = MPNet(h).to(dev); m.load_state_dict(torch.load(path, map_location=dev, weights_only=False)['generator']); return m.eval()
def mp_forward(m, wav):                       # wav [B, T] (RMS-normalised) -> enhanced [B, T]
    amp, pha, _ = mag_pha_stft(wav, h.n_fft, h.hop_size, h.win_size, h.compress_factor)
    amp_g, pha_g, com_g = m(amp, pha)
    return mag_pha_istft(amp_g, pha_g, h.n_fft, h.hop_size, h.win_size, h.compress_factor), (amp_g, pha_g, com_g)
@torch.no_grad()
def enhance_mp(m, x, win=4.0, hop=3.0):
    x = np.asarray(x, np.float32); n = len(x); sr = 16000
    nf = float(np.sqrt(n / (np.sum(x**2) + 1e-9))); xn = torch.from_numpy(x * nf).to(dev)
    W, H = int(win * sr), int(hop * sr)
    if n <= W:
        y = mp_forward(m, xn[None])[0][0][:n]
    else:
        y = torch.zeros(n, device=dev); wsum = torch.zeros(n, device=dev)
        fade = torch.ones(W, device=dev); r = W - H; fade[:r] = torch.linspace(0, 1, r, device=dev); fade[-r:] = torch.linspace(1, 0, r, device=dev)
        s = 0
        while True:
            e = min(s + W, n); seg = xn[s:e]
            if len(seg) < W: seg = torch.nn.functional.pad(seg, (0, W - len(seg)))
            out = mp_forward(m, seg[None])[0][0][:W]
            L = e - s; f = fade[:L].clone()
            if s == 0: f[:r] = 1
            if e == n: f[-min(r, L):] = 1
            y[s:e] += out[:L] * f; wsum[s:e] += f
            if e == n: break
            s += H
        y = y / wsum.clamp_min(1e-6)
    y = (y / nf).cpu().numpy()
    return np.pad(y, (0, max(0, n - len(y))))[:n].astype(np.float32)
mp = load_mp(); print('MP-SENet params (M):', round(sum(p.numel() for p in mp.parameters()) / 1e6, 2))

In [ ]:
# Isolated validation noise events + fingerprint clustering
SR = 16000
NV = Path('/kaggle/working/bank/val_noise'); NV.mkdir(parents=True, exist_ok=True)
segs, fps, owner = [], [], []
for v in items:
    if not v['syn']: continue
    y = data.read_audio(v['path']); c = data.read_audio(v['ref']); L = min(len(y), len(c)); n = (y[:L] - c[:L]).astype(np.float32)
    for k, e in enumerate(meta.get(v['id'], {}).get('NoiseSubCategoryTimeStamp', [])):
        a, b = max(0, int((float(e['start']) - 0.02) * SR)), min(L, int((float(e['end']) + 0.02) * SR))
        if b - a < int(0.3 * SR) or np.abs(n[a:b]).max() < 1e-4: continue
        seg = n[a:b]; p = NV / f"{v['id'][-40:]}_{k}.wav"; sf.write(p, seg, SR)
        mel = librosa.power_to_db(librosa.feature.melspectrogram(y=seg, sr=SR, n_mels=48, n_fft=512, hop_length=160) + 1e-10)
        f = librosa.util.fix_length(mel, size=48, axis=1).ravel(); f = (f - f.mean()) / (f.std() + 1e-6)
        segs.append({'path': str(p), 'dur': (b - a) / SR}); fps.append(f / np.linalg.norm(f)); owner.append(v['id'])
F = np.stack(fps); cents, lab = [], np.zeros(len(F), int)
for i, f in enumerate(F):
    if cents:
        s = np.stack(cents) @ f; j = int(s.argmax())
        if s[j] > 0.95: lab[i] = j; continue
    cents.append(f); lab[i] = len(cents) - 1
ncl = len(cents); rng = np.random.default_rng(7)
print('noise events', len(F), '| clusters', ncl)
for frac in [0.35, 0.45, 0.55]:
    held = set(rng.permutation(ncl)[:int(frac * ncl)].tolist())
    clip_cl = {}
    for o, l in zip(owner, lab): clip_cl.setdefault(o, set()).add(int(l))
    HELD_CLIPS = [o for o, s in clip_cl.items() if s <= held]
    if len(HELD_CLIPS) >= 80: break
train_noise = [s for s, l in zip(segs, lab) if int(l) not in held]
print(f'held-out clusters {len(held)}/{ncl} | held-out eval clips {len(HELD_CLIPS)} | train noise segments {len(train_noise)}')

In [ ]:
# ESC-50 noise segments (active part, 0.3-2.5 s), resampled to 16 kHz
E = Path('/kaggle/working/ESC-50-master'); NE = Path('/kaggle/working/bank/esc50'); NE.mkdir(parents=True, exist_ok=True)
import pandas as pd
esc = pd.read_csv(E/'meta/esc50.csv'); esc_noise = []
for _, r in esc.iterrows():
    w, _ = librosa.load(E/'audio'/r.filename, sr=SR, mono=True)
    w, _ = librosa.effects.trim(w, top_db=30)
    if len(w) < int(0.3 * SR): continue
    w = w[:int(2.5 * SR)].astype(np.float32); p = NE / r.filename; sf.write(p, w, SR)
    esc_noise.append({'path': str(p), 'dur': len(w) / SR, 'cat': r.category})
print('ESC-50 segments', len(esc_noise), '| categories', esc.category.nunique())

In [ ]:
# MUSAN noise from the uploaded Kaggle dataset (no download)
MUF = sorted(glob.glob('/kaggle/input/**/free-sound/*.wav', recursive=True) + glob.glob('/kaggle/input/**/sound-bible/*.wav', recursive=True))
print('MUSAN wav files found:', len(MUF), '(expect 930)'); assert len(MUF) > 800, 'attach the musan-noise dataset'
NM = Path('/kaggle/working/bank/musan'); NM.mkdir(parents=True, exist_ok=True); rngm = np.random.default_rng(3); musan_noise = []
for k, fpath in enumerate(MUF):
    w, _ = librosa.load(fpath, sr=SR, mono=True)
    if len(w) < int(0.5 * SR) or np.abs(w).max() < 1e-4: continue
    for j in range(min(4, max(1, len(w) // (3 * SR)))):
        L = int(rngm.uniform(0.5, 2.5) * SR); st = int(rngm.integers(0, max(1, len(w) - L)))
        seg = w[st:st + L].astype(np.float32)
        if np.sqrt((seg**2).mean()) < 1e-3: continue
        p = NM / f'{k:04d}_{j}.wav'; sf.write(p, seg, SR); musan_noise.append({'path': str(p), 'dur': len(seg) / SR})
print('MUSAN noise segments:', len(musan_noise))

In [ ]:
# Noise augmentation: speed perturbation, random EQ, synthetic reverb -> one augmented copy per base segment
from scipy.signal import butter, sosfilt, fftconvolve
NA = Path('/kaggle/working/bank/aug'); NA.mkdir(parents=True, exist_ok=True); rnga = np.random.default_rng(11)
def augment(w):
    f = rnga.choice([0.9, 1.0, 1.1]);  w = librosa.resample(w, orig_sr=int(SR * f), target_sr=SR) if f != 1.0 else w
    if rnga.random() < 0.6:
        if rnga.random() < 0.5: sos = butter(4, rnga.uniform(2000, 7000), 'low', fs=SR, output='sos')
        else: sos = butter(2, rnga.uniform(100, 500), 'high', fs=SR, output='sos')
        w = sosfilt(sos, w)
    if rnga.random() < 0.5:
        rt = rnga.uniform(0.15, 0.7); t = np.arange(int(rt * SR)) / SR
        rir = rnga.standard_normal(len(t)) * np.exp(-6.9 * t / rt); rir[0] = 1.0
        w = fftconvolve(w, rir / np.abs(rir).sum() * 4)[:len(w)]
    pk = np.abs(w).max(); return (w / pk * 0.9 if pk > 0 else w).astype(np.float32)
base = train_noise + esc_noise + musan_noise; aug_noise = []
for k, nb_ in enumerate(base):
    w, _ = sf.read(nb_['path'], dtype='float32'); w = augment(w)
    if len(w) < int(0.3 * SR): continue
    p = NA / f'a{k:05d}.wav'; sf.write(p, w, SR); aug_noise.append({'path': str(p), 'dur': len(w) / SR})
print('noise pool:', len(base), 'base +', len(aug_noise), 'augmented')

In [ ]:
import gc, zipfile, subprocess
D_CK = [x for x in glob.glob('/kaggle/input/**/*anneal*.pt', recursive=True) if os.path.isfile(x)][0]
mD = load_mp(D_CK); globals().pop('mp', None); gc.collect(); torch.cuda.empty_cache(); print('D:', D_CK)

In [ ]:
# 2) CHECK: how much does D alter Vaani-native audio? SI-SDR(reference, D(reference)) on validation refs and natural clips (higher = D leaves it alone)
byid = {v['id']: v for v in items}
def alter(p): r = data.read_audio(p); return float(evaluate.si_sdr(r, enhance_mp(mD, r, win=8.0, hop=6.0)))
refs = [v['ref'] for v in items if v['syn']][:200]; nats = [v['path'] for v in items if not v['syn']][:300]
a_ref = np.array([alter(p) for p in refs]); a_nat = np.array([alter(p) for p in nats])
print(f'D on VAL synthetic REFERENCES: SI-SDR(ref, D(ref)) p25 {np.percentile(a_ref,25):.1f} | p50 {np.percentile(a_ref,50):.1f} | p75 {np.percentile(a_ref,75):.1f} dB')
print(f'D on VAL natural clips       : SI-SDR(x, D(x))     p25 {np.percentile(a_nat,25):.1f} | p50 {np.percentile(a_nat,50):.1f} | p75 {np.percentile(a_nat,75):.1f} dB')
print('If test references are NOISY natural clips, a model that strips their native noise is penalised; E13 trains D to remove only library events.')

In [ ]:
# FREE CHECK: does 'natural clip + library events (target = natural clip)' reproduce the LB (~13 dB) for D? (val synthetic gives ~18 dB locally)
import pandas as pd
from inference import Transcriber
escm = pd.read_csv('/kaggle/working/ESC-50-master/meta/esc50.csv').set_index('filename')
def esc_fold(p): f = os.path.basename(p); return int(escm.loc[f]['fold']) if f in escm.index else 0
ev_eval = [e for e in esc_noise if esc_fold(e['path']) == 5] + musan_noise[2100:]
def rd(p): w, _ = sf.read(p, dtype='float32'); return w if w.ndim == 1 else w.mean(1)
rng = np.random.RandomState(0); NAT_ALL = [v['id'] for v in items if not v['syn']]
NS = sorted(NAT_ALL[i] for i in rng.choice(len(NAT_ALL), 600, replace=False)); RNG = np.random.default_rng(9); LX, LT = {}, {}
for i in NS:
    s = data.read_audio(byid[i]['path']); L = len(s); ps = float((s**2).mean()) + 1e-10; x = s.copy()
    for _ in range(int(RNG.choice([1, 1, 2]))):
        e = rd(ev_eval[RNG.integers(len(ev_eval))]['path'])[:int(2.5 * SR)]
        if len(e) < int(0.3 * SR) or len(e) >= L: continue
        on = int(RNG.integers(0, L - len(e))); x[on:on + len(e)] += e * np.sqrt(ps / ((e**2).mean() + 1e-10) / 10**(RNG.uniform(-20, 5) / 10))
    LX[i], LT[i] = x.astype(np.float32), s.astype(np.float32)
def post(x, y):
    ri, ro = np.sqrt((x**2).mean()), np.sqrt((y**2).mean()); z = y * (ri / ro) if ro > 1e-8 else y
    pk = np.abs(z).max(); return (z * (0.99 / pk) if pk > 0.99 else z).astype(np.float32)
def tta(x, shifts=(0, 50)): return np.mean([enhance_mp(mD, np.pad(x, (s_, 0)), win=8.0, hop=6.0)[s_:s_ + len(x)] for s_ in shifts], 0).astype(np.float32)
si_in = float(np.mean([evaluate.si_sdr(LT[i], LX[i]) for i in NS]))
out = {i: post(LX[i], tta(LX[i])) for i in NS}; si_D = float(np.mean([evaluate.si_sdr(LT[i], out[i]) for i in NS]))
si_keep = float(np.mean([evaluate.si_sdr(LT[i], LX[i] - (LX[i] - out[i])) for i in NS]))
H = [i for i in HELD_CLIPS if i in byid]; si_val = float(np.mean([evaluate.si_sdr(data.read_audio(byid[i]['ref']), post(data.read_audio(byid[i]['path']), tta(data.read_audio(byid[i]['path'])))) for i in H]))
print(f'natural + library events: input SI-SDR {si_in:.2f} | D output SI-SDR {si_D:.2f}   <- compare with LB 13.18')
print(f'validation synthetic held-out: D output SI-SDR {si_val:.2f}                 <- local reference 17.9')
asr = Transcriber(HF); GT = {i: evaluate.normalize(byid[i]['gt']) for i in NS}
def pooled(hyp): k_ = [i for i in NS if GT[i]]; return jwer([GT[i] for i in k_], [evaluate.normalize(hyp.get(i, '')) or '@' for i in k_])
def asr_all(sig):
    hyp = {}
    for j in range(0, len(NS), 16):
        for i, t in zip(NS[j:j+16], asr.transcribe_many([sig[i] for i in NS[j:j+16]])): hyp[i] = t
    return hyp
w_in, w_out, w_ref = pooled(asr_all(LX)), pooled(asr_all(out)), pooled(asr_all(LT))
print(f'WER: input {w_in:.4f} | D output {w_out:.4f} | reference (natural clip) {w_ref:.4f} -> dWER(D) {100*(w_in-w_out):+.2f} | ceiling (output=reference) {100*(w_in-w_ref):+.2f}')